In [ ]:
# # Decouplerpy install
# conda create -n decouplerpy --offline
# conda activate decouplerpy
# conda install -c conda-forge decoupler-py scanpy
# conda install -c conda-forge jupyterlab
# python -m ipykernel install --user --name=decouplerpy --display-name='Environment (decouplerpy)'

In [2]:
import scanpy as sc
import decoupler as dc
import numpy as np
import pickle
import pandas as pd
import anndata
import scipy.sparse as sp
import os
import gc

# Plotting options, change to your liking
# sc.settings.set_figure_params(dpi=200, frameon=False)
# sc.set_figure_params(dpi=200)
sc.set_figure_params(figsize=(4, 4))

In [3]:
with open('go_term_accession_name_20250203.pkl','rb') as file:
    go_terms_dict = pickle.load(file)

# https://github.com/bioinplant/PCmaster/blob/development-1/Tmp_tutorial/go_term_accession_name_20250203.pkl
# https://pan.baidu.com/s/1xdaHWlLQcYXR6uEsx2jv-w?pwd=46ny

In [4]:
def print_head(obj, n=5):
    if isinstance(obj, (list, set, np.ndarray)):
        print(list(obj)[:n])
    elif isinstance(obj, dict):
        print({k: obj[k] for k in list(obj)[:n]})
    elif isinstance(obj, pd.DataFrame):
        print(obj.head(n))
    else:
        print("Unsupported object type.")

my_list = [i for i in range(10)]
my_set = {i for i in range(10)}
my_dict = {f"key_{i}": i for i in range(10)}
my_df = pd.DataFrame({"A": range(10), "B": range(10, 20)})
my_array = np.array([1, 2, 3, 4, 5])

print_head(my_list)
print_head(my_set)
print_head(my_dict)
print_head(my_df)
print_head(my_array)

[0, 1, 2, 3, 4]
[0, 1, 2, 3, 4]
{'key_0': 0, 'key_1': 1, 'key_2': 2, 'key_3': 3, 'key_4': 4}
   A   B
0  0  10
1  1  11
2  2  12
3  3  13
4  4  14
[1, 2, 3, 4, 5]


In [5]:
print_head(go_terms_dict)

{'GO:0009706': 'GO:0009706_chloroplast inner membrane', 'GO:0009658': 'GO:0009658_chloroplast organization', 'GO:0003729': 'GO:0003729_mRNA binding', 'GO:0015120': 'GO:0015120_phosphoglycerate transmembrane transporter activity', 'GO:0015121': 'GO:0015121_phosphoenolpyruvate:phosphate antiporter activity'}


In [6]:
with open('pathway_dbs_2.pkl','rb') as file:
    dbs = pickle.load(file)

# https://github.com/bioinplant/PCmaster/blob/development-1/Tmp_tutorial/pathway_dbs_2.pkl
# https://pan.baidu.com/s/1rNWEEreZm8PdNFQFdQpjcQ?pwd=4hs2

In [7]:
for db in dbs:
    print(type(db))
    print(db.head())
    break

<class 'pandas.core.frame.DataFrame'>
     genesymbol                                       geneset
0  KAI9215444.1  GO:0033014_tetrapyrrole biosynthetic process
1  KAI9215445.1  GO:0033014_tetrapyrrole biosynthetic process
2  KAI9215671.1  GO:0033014_tetrapyrrole biosynthetic process
3  KAI9215672.1  GO:0033014_tetrapyrrole biosynthetic process
4  KAI9377244.1  GO:0033014_tetrapyrrole biosynthetic process


In [8]:
def get_h5ad_db_intersection(
    directory,dbs,
    # prjset=set('blank.h5ad'),
):
    redict = {}
    for root, dirs, files in os.walk(directory):
        for filename in files:
            if filename.endswith(".h5ad"):
                # count = 0
                # for prj in prjset:
                #     if prj.strip() in filename:
                #         print(f"{filename} done, pass")
                #         count = 1
                #         break
                # if count == 1:
                #     print("-" * 40)
                #     continue
                file_path = os.path.join(root, filename)
                print(f"{file_path} preprocessing")
                try:
                    adata = sc.read(file_path)
                    print(adata.var_names)
                    tmplist1 = []
                    for i in adata.var_names:
                        tmplist1.append(str(i))
                    for tmpdb in dbs:
                        tmplist2 = tmpdb['genesymbol'].tolist()
                        if len(set(tmplist1) & set(tmplist2)) > 10:
                            redict[file_path] = tmpdb
                            # break
                            print(f"file path: {file_path}")
                            print_head(tmpdb)
                            break
                except Exception as error:
                    print(error)
                print("-" * 40)
    return redict

In [9]:
h5ad_db_dict = get_h5ad_db_intersection(
    directory='./data-files-tmp/', dbs=dbs,
)

./data-files-tmp/PRJNA740934.h5ad preprocessing
Index(['Zm00001d028471', 'Zm00001d053828', 'Zm00001d038226', 'Zm00001d036175',
       'Zm00001d045877', 'Zm00001d035605', 'Zm00001d024273', 'Zm00001d023559',
       'Zm00001d036535', 'Zm00001d046865',
       ...
       'Zm00001d034383', 'Zm00001d005856', 'Zm00001d031677', 'Zm00001d007824',
       'Zm00001d038051', 'Zm00001d010713', 'Zm00001d046423', 'Zm00001d025375',
       'Zm00001d026020', 'Zm00001d012842'],
      dtype='object', length=3853)
file path: ./data-files-tmp/PRJNA740934.h5ad
        genesymbol                                            geneset
0   Zm00001d024935  GO:0000380_alternative mRNA splicing, via spli...
2   Zm00001d011652  GO:0000380_alternative mRNA splicing, via spli...
14  Zm00001d012060  GO:0000380_alternative mRNA splicing, via spli...
19  Zm00001d004546  GO:0000380_alternative mRNA splicing, via spli...
20  Zm00001d043745  GO:0000380_alternative mRNA splicing, via spli...
--------------------------------------

/mnt/sda/dhzheng/miniconda3/envs/decouplerpy/lib/python3.12/site-packages/anndata/compat/__init__.py:358: FutureWarning: Moving element from .uns['neighbors']['distances'] to .obsp['distances'].

This is where adjacency matrices should go now.
  warn(


In [10]:
def dcp_aucell(adata,godb):
    print(adata)
    if adata.X.max() > 14:
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)
        print('Data normalized.')
    try:
        expression_df = pd.DataFrame(
            adata.X.toarray() if not isinstance(adata.X, pd.DataFrame) else adata.X,
            index=adata.obs_names,
            columns=adata.var_names,
        )
    except Exception as error:
        pass
    try:
        expression_df = pd.DataFrame(
            adata.X,
            index=adata.obs_names,
            columns=adata.var_names,
        )
    except Exception as error:
        pass
    acts = dc.run_aucell( #####!!!!!
        mat=expression_df,
        net=godb,
        source='geneset',
        target='genesymbol',
        verbose=True,
        min_n=1,
        # weight=None,
    )
    score = acts[0]
    print(score.head())
    new_adata = anndata.AnnData(X=score.values, obs=pd.DataFrame(index=score.index), var=pd.DataFrame(index=score.columns))
    new_adata.obs = adata.obs
    print(new_adata)
    print(new_adata.X.max())
    print(new_adata.X.min())
    print('aucell finished\n')
    return new_adata

In [11]:
def dcp_ulm(adata,godb):
    print(adata)
    if adata.X.max() > 14:
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)
        print('Data normalized.')
    try:
        expression_df = pd.DataFrame(
            adata.X.toarray() if not isinstance(adata.X, pd.DataFrame) else adata.X,
            index=adata.obs_names,
            columns=adata.var_names,
        )
    except Exception as error:
        pass
    try:
        expression_df = pd.DataFrame(
            adata.X,
            index=adata.obs_names,
            columns=adata.var_names,
        )
    except Exception as error:
        pass
    acts = dc.run_ulm( #####!!!!!
        mat=expression_df,
        net=godb,
        source='geneset',
        target='genesymbol',
        verbose=True,
        min_n=1,
        weight=None,
    )
    score = acts[0]
    print(score.head())
    new_adata = anndata.AnnData(X=score.values, obs=pd.DataFrame(index=score.index), var=pd.DataFrame(index=score.columns))
    new_adata.obs = adata.obs
    print(new_adata)
    print(new_adata.X.max())
    print(new_adata.X.min())
    print('ulm finished\n')
    return new_adata

In [12]:
def process_h5ad_files_aucell(
    directory, h5ad_db_dict, 
    # prjset=set('blank.h5ad'),
):
    for root, dirs, files in os.walk(directory):
        for filename in files:
            if filename.endswith(".h5ad"):
                # count = 0
                # for prj in prjset:
                #     if prj.strip() in filename:
                #         print(f"{filename} done, pass")
                #         count = 1
                #         break
                # if count == 1:
                #     print("-" * 40)
                #     continue
                file_path = os.path.join(root, filename)
                save_path = os.path.join(root, filename.replace('.h5ad','-aucell.h5ad'))
                if os.path.exists(save_path):
                    print(f'{save_path} exists')
                    continue
                print(f"file path: {file_path}")
                if file_path in h5ad_db_dict:
                    try:
                        godb = h5ad_db_dict[file_path]
                        adata = sc.read(file_path)
                        del adata.raw
                        the_min = adata.X.min()
                        if the_min < 0:
                            adata.X = adata.X - the_min + 1.0
                        new_adata = dcp_aucell(adata,godb)
                        new_adata.write(save_path)
                        new_adata = sc.read(save_path)
                        print(new_adata)
                        print(new_adata.obs_names)
                        print(new_adata.var_names)
                    except Exception as error:
                        print(file_path)
                        print('error!')
                        print(error)
                print("-" * 40)

In [13]:
def process_h5ad_files_ulm(
    directory, h5ad_db_dict, 
    # prjset=set('blank.h5ad'),
):
    for root, dirs, files in os.walk(directory):
        for filename in files:
            if filename.endswith(".h5ad"):
                # count = 0
                # for prj in prjset:
                #     if prj.strip() in filename:
                #         print(f"{filename} done, pass")
                #         count = 1
                #         break
                # if count == 1:
                #     print("-" * 40)
                #     continue
                file_path = os.path.join(root, filename)
                save_path = os.path.join(root, filename.replace('.h5ad','-ulm.h5ad'))
                if os.path.exists(save_path):
                    print(f'{save_path} exists')
                    continue
                print(f"file path: {file_path}")
                if file_path in h5ad_db_dict:
                    try:
                        godb = h5ad_db_dict[file_path]
                        adata = sc.read(file_path)
                        del adata.raw
                        the_min = adata.X.min()
                        if the_min < 0:
                            adata.X = adata.X - the_min + 1.0
                        new_adata = dcp_ulm(adata,godb)
                        new_adata.write(save_path)
                        new_adata = sc.read(save_path)
                        print(new_adata)
                        print(new_adata.obs_names)
                        print(new_adata.var_names)
                    except Exception as error:
                        print(file_path)
                        print('error!')
                        print(error)
                print("-" * 40)

In [14]:
process_h5ad_files_aucell(
    directory='./data-files-tmp/',h5ad_db_dict=h5ad_db_dict,
)

file path: ./data-files-tmp/PRJNA740934.h5ad


/mnt/sda/dhzheng/miniconda3/envs/decouplerpy/lib/python3.12/site-packages/anndata/compat/__init__.py:358: FutureWarning: Moving element from .uns['neighbors']['distances'] to .obsp['distances'].

This is where adjacency matrices should go now.
  warn(


AnnData object with n_obs × n_vars = 33388 × 3853
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'nCount_SCT', 'nFeature_SCT', 'integrated_snn_res.0.5', 'seurat_clusters', 'celltype', 'celltype_after', 'celltype_before'
    var: 'features', 'SCT_features'
    uns: 'neighbors'
    obsm: 'X_pca', 'X_tsne', 'X_umap'
    varm: 'PCs'
    layers: 'SCT'
    obsp: 'distances'
Data normalized.
Running aucell on mat with 33388 samples and 3853 targets for 235 sources.


100%|██████████| 33388/33388 [01:25<00:00, 389.73it/s]


source                GO:0000055_ribosomal large subunit export from nucleus  \
AAACCTGAGACAAGCC-1_1                                                0.0        
AAACCTGAGCTGTCTA-1_1                                                0.0        
AAACCTGCAATAGAGT-1_1                                                0.0        
AAACCTGGTAACGCGA-1_1                                                0.0        
AAACCTGGTCTCGTTC-1_1                                                0.0        

source                GO:0000145_exocyst  \
AAACCTGAGACAAGCC-1_1                 0.0   
AAACCTGAGCTGTCTA-1_1                 0.0   
AAACCTGCAATAGAGT-1_1                 0.0   
AAACCTGGTAACGCGA-1_1                 0.0   
AAACCTGGTCTCGTTC-1_1                 0.0   

source                GO:0000293_ferric-chelate reductase activity  \
AAACCTGAGACAAGCC-1_1                                           0.0   
AAACCTGAGCTGTCTA-1_1                                           0.0   
AAACCTGCAATAGAGT-1_1                        

In [15]:
process_h5ad_files_ulm(
    directory='./data-files-tmp/',h5ad_db_dict=h5ad_db_dict,
)

file path: ./data-files-tmp/PRJNA740934.h5ad


/mnt/sda/dhzheng/miniconda3/envs/decouplerpy/lib/python3.12/site-packages/anndata/compat/__init__.py:358: FutureWarning: Moving element from .uns['neighbors']['distances'] to .obsp['distances'].

This is where adjacency matrices should go now.
  warn(


AnnData object with n_obs × n_vars = 33388 × 3853
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'nCount_SCT', 'nFeature_SCT', 'integrated_snn_res.0.5', 'seurat_clusters', 'celltype', 'celltype_after', 'celltype_before'
    var: 'features', 'SCT_features'
    uns: 'neighbors'
    obsm: 'X_pca', 'X_tsne', 'X_umap'
    varm: 'PCs'
    layers: 'SCT'
    obsp: 'distances'
Data normalized.
Running ulm on mat with 33388 samples and 3853 targets for 235 sources.
                      GO:0000055_ribosomal large subunit export from nucleus  \
AAACCTGAGACAAGCC-1_1                                          -0.120275        
AAACCTGAGCTGTCTA-1_1                                          -0.116518        
AAACCTGCAATAGAGT-1_1                                          -0.116213        
AAACCTGGTAACGCGA-1_1                                          -0.118633        
AAACCTGGTCTCGTTC-1_1                                          -0.120753        

                      GO:0000145_exocyst  \
AAACCTGAG